The script will take in the bronze tables to create the master tables for each path, 

First Gather all scema information

In [0]:
from pyspark.sql.functions import col, to_date
import pandas as pd

data_timeline = "202607"                # to use in the table names, adjust as needed
# create the silver schema beforehand
silver_schema_path = "2026_july_balance.silver_schema"

# gather other schemas
bronze_schema_path = "`2026_july_balance`.bronze_schema"


all_bronze_tables = spark.sql(f"SHOW TABLES IN {bronze_schema_path}").toPandas()["tableName"].tolist()
fund_names = sorted(set(name.split("_")[2] for name in all_bronze_tables))
print(fund_names)



Second Explore Bronze Table Details

In [0]:
import pandas as pd

# put accounts tables and dimensions in a dataframe
def get_table_dimensions(table_name):
    target_tables = [x for x in all_bronze_tables if x.startswith(table_name)]
    table_dict = {}
    for table in target_tables:
        df = spark.table(f"{bronze_schema_path}.{table}")
        table_dict[table] = {"columns": len(df.columns), "rows": df.count()}
    return table_dict


In [0]:
# get the column and row counts in the files in a dataframe
account_dict = get_table_dimensions("accounts")
account_df = pd.DataFrame.from_dict(account_dict, orient="index", columns=["columns", "rows"])

balance_dict = get_table_dimensions("balances")
balance_df = pd.DataFrame.from_dict(balance_dict, orient="index", columns=["columns", "rows"])

project_dict = get_table_dimensions("projects")
project_df = pd.DataFrame.from_dict(project_dict, orient="index", columns=["columns", "rows"])

funds_dict = get_table_dimensions("funds")
funds_df = pd.DataFrame.from_dict(funds_dict, orient="index", columns=["columns", "rows"])

department_dict = get_table_dimensions("departments")
department_df = pd.DataFrame.from_dict(department_dict, orient="index", columns=["columns", "rows"])


In [0]:
print(account_df)
print(balance_df)
print(project_df)
print(funds_df)
print(department_df)

Third, define the Function to process the Code Descriptions in the silver tables

In [0]:
from pyspark.sql.functions import col

def join_code_desc(silver_df, code_table_name):
  # Define fully qualified table names
  
  code_full_path = f"workspace.code_desc_project_master.{code_table_name}"

  # Read the tables to check columns
  
  code_df = spark.table(code_full_path)

  join_column, add_columns = code_df.columns[0], code_df.columns[1:]

  if join_column in silver_df.columns:
      
      try: 
        # Perform join (will have both join columns)
        result_df = silver_df.alias("stn").join(
        code_df.alias("cdt"),
        on=col(f"stn.{join_column}") == col(f"cdt.{join_column}"),
        how="left").select(
          *[col(f"stn.{c}") for c in silver_df.columns],  # all silver cols with explicit alias
          *[col(f"cdt.{c}") for c in add_columns])

        print(f"Joined {code_table_name} tables")
        print(f"Result row counts: {result_df.count()}, column counts: {len(result_df.columns)}")
        # display(result_df)
        return result_df

      except Exception as e:
          print(f"Failed to join: {e}")
  else:
      print(f"Join column '{join_column}' not found in {table_name} table")

Finally, loop through the table and join the fund data tables and apply the Code Description function to add the code descriptions. At the end, commit the tables as they are created.

In [0]:
for fund in fund_names:
    print(f"processing {fund}")

    try:
        
        balance = spark.table(f"{bronze_schema_path}.balances_{data_timeline}_{fund}")
        balance = balance.withColumn("Effective_Date", to_date(col("Effective_Date"), "yyyyMMdd"))
        balance = balance.withColumn("Amount", col("Amount").cast("double"))
        
        # print(f"{fund} balance table has {balance.count()} records")
        
        # Start joining tables together...
        master = balance            #set balance the master table

        # Join accounts table
        try:
            accounts = spark.table(f"{bronze_schema_path}.accounts_{data_timeline}_{fund}")
            # print(f"{fund} accounts table has {accounts.count()} records")
            accounts_cols = [col for col in accounts.columns if col not in balance.columns or col in ['Account_Code', 'Entity_Code']]
            master = master.join(accounts.select(*accounts_cols), on=['Account_Code', 'Entity_Code'], how="left")
            # print(f"Done joining accounts table for {fund}")

        except:
            print(f"{fund} accounts table not found")


        # Join departments table - parent department descriptions are available so pull them in
        try:
            departments = spark.table(f"{bronze_schema_path}.departments_{data_timeline}_{fund}")
            # print(f"{fund} departments table has {departments.count()} records")
            
            # Get the parent department descriptions
            dept_parent = departments.select(col("Entity_Code"), col("Department_Code"), col("Department_Description").alias("Parent_Dept_Description")) 
            departments_w_parent = departments.join(dept_parent, on =
                                            (departments.Entity_Code == dept_parent.Entity_Code) &
                                            (departments.Parent_Department_Code  == dept_parent.Department_Code),
                                            how="left").drop(dept_parent.Entity_Code, dept_parent.Department_Code)
            
            # Now join with master
            departments_cols = [col for col in departments_w_parent.columns if col not in balance.columns or col in ['Department_Code', 'Entity_Code']]
            master = master.join(departments_w_parent.select(*departments_cols), on=['Department_Code', 'Entity_Code'], how="left")
            # print(f"Done joining departments table for {fund}")

        except:
            print(f"{fund} departments table not found")

        
        # Join funds table. - parent fund codes are not of the same type.
        try:
            funds_data = spark.table(f"{bronze_schema_path}.funds_{data_timeline}_{fund}")
            # print(f"{fund} funds table has {funds_data.count()} records")
            funds_cols = [col for col in funds_data.columns if col not in balance.columns or col in ['Fund_Code', 'Entity_Code']]
            master = master.join(funds_data.select(*funds_cols), on=['Fund_Code', 'Entity_Code'], how="left")
            # print(f"Done joining funds table for {fund}")

        except:
            print(f"{fund} funds table not found")
        
        # Join projects table
        try:
            projects = spark.table(f"{bronze_schema_path}.projects_{data_timeline}_{fund}")
            # print(f"{fund} projects table has {projects.count()} records")
            projects = (projects.withColumn("Project_Start_Date", to_date(col("Project_Start_Date"), "yyyyMMdd"))
                                .withColumn("Project_End_Date", to_date(col("Project_End_Date"), "yyyyMMdd"))
            )

            project_cols = [col for col in projects.columns if col not in balance.columns or col in ['Project_Code', 'Entity_Code']]

            master = master.join(projects.select(*project_cols), on=['Project_Code', 'Entity_Code'], how="left")
            # print(f"Done joining projects table for {fund}")

        except:
            print(f"{fund} projects table not found")

        print(f"{fund} silver table created with {master.count()} records against {balance.count()} balance records and {len(master.columns)} columns")

# Add parent account code description since the join
        try:

    # first read the parent account code table as a delta table since the number of columns to join is different from other code desc tables.
            pac = spark.table(f"workspace.code_desc_project_master.parent_account_codes").select("Parent_Account_Code",
            col("Account_Code_Description").alias("Parent_Account_Code_Description"),
            col("Account_type").alias("Parent_Account_Code_Type"),
        )
            master = master.join(pac, on="Parent_Account_Code", how="left")
        except Exception as e:
            print(f"Parent account code description had {e} for {fund}")


        # entity_code_table = workspace.code_desc_project_master.entity_code_desc
        master = join_code_desc(silver_df=master, code_table_name="entity_code_desc")

        # function_code_table = "function_code_desc"
        master = join_code_desc(silver_df=master, code_table_name="function_code_desc")

        # program_code_table = "program_code_desc"
        master = join_code_desc(silver_df=master, code_table_name="program_code_desc")

        # Save master table
        master_table_name = f"{silver_schema_path}.{fund}_master"
        master.write.format("delta") \
              .mode("overwrite") \
              .option("overwriteSchema", "true") \
              .saveAsTable(master_table_name)

        print(f"Created {master_table_name} table")
        
    except Exception as e:
        print(f"Failed to access {fund}: {e}")

print("Silver layer created and uploaded")
  